Optimize feature for a classification problem:1 RFF 2 Random Feature

In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.feature_selection import RFE
from sklearn.metrics import accuracy_score, classification_report
from sklearn.linear_model import LogisticRegression

try:
    df = pd.read_csv('https://raw.githubusercontent.com/datasciencedojo/datasets/master/titanic.csv')
    print("Titanic dataset loaded successfully.")
except Exception as e:
    print(f"Error loading Titanic dataset: {e}")
    exit()

processed_df = df.copy()
processed_df.drop(['PassengerId', 'Name', 'Ticket', 'Cabin'], axis=1, inplace=True)
processed_df['Age'] = processed_df['Age'].fillna(processed_df['Age'].median())
processed_df['Fare'] = processed_df['Fare'].fillna(processed_df['Fare'].median())
processed_df['Embarked'] = processed_df['Embarked'].fillna(processed_df['Embarked'].mode()[0])
processed_df = pd.get_dummies(processed_df, columns=['Sex', 'Embarked'], drop_first=True)

X = processed_df.drop('Survived', axis=1)
y = processed_df['Survived']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
print(f"\nTraining data shape: {X_train.shape}, Testing data shape: {X_test.shape}")
print(f"Target data shape: {y_train.shape}, Target test data shape: {y_test.shape}")

print("\n--- Step 3: Using Random Forest for Feature Importance ---")
rf_model_for_importance = RandomForestClassifier(n_estimators=100, random_state=42)
rf_model_for_importance.fit(X_train, y_train)
feature_importances = pd.Series(rf_model_for_importance.feature_importances_, index=X_train.columns)
sorted_feature_importances = feature_importances.sort_values(ascending=False)
print("\nFeature Importances from Random Forest (Top 10):")
print(sorted_feature_importances.head(10))

print("\n--- Step 4: Applying Recursive Feature Elimination (RFE) ---")
n_features_to_select = 6
rfe_selector = RFE(estimator=LogisticRegression(max_iter=1000, random_state=42), n_features_to_select=n_features_to_select, step=1)
rfe_selector.fit(X_train, y_train)
selected_features_rfe_mask = rfe_selector.support_
selected_features_rfe = X_train.columns[selected_features_rfe_mask].tolist()
print("\n--- Submission Requirement: Selected Features ---")
print(f"Selected features by RFE: {selected_features_rfe}")

print("\n--- Step 5: Training Models and Comparing Performance ---")

print("\nTraining model with ALL features...")
model_all_features = RandomForestClassifier(n_estimators=100, random_state=42)
model_all_features.fit(X_train, y_train)
y_pred_all_features = model_all_features.predict(X_test)
accuracy_all_features = accuracy_score(y_test, y_pred_all_features)
print(f"Accuracy with ALL features: {accuracy_all_features:.4f}")
print("\nClassification Report (ALL features):")
print(classification_report(y_test, y_pred_all_features))

print("\nTraining model with SELECTED features...")
X_train_selected = X_train[selected_features_rfe]
X_test_selected = X_test[selected_features_rfe]
model_selected_features = RandomForestClassifier(n_estimators=100, random_state=42)
model_selected_features.fit(X_train_selected, y_train)
y_pred_selected_features = model_selected_features.predict(X_test_selected)
accuracy_selected_features = accuracy_score(y_test, y_pred_selected_features)
print(f"Accuracy with SELECTED features: {accuracy_selected_features:.4f}")
print("\nClassification Report (SELECTED features):")
print(classification_report(y_test, y_pred_selected_features))

print(f"Comparison: Accuracy (All Features) = {accuracy_all_features:.4f} vs. Accuracy (Selected Features) = {accuracy_selected_features:.4f}")

if accuracy_selected_features > accuracy_all_features:
    print("\nObservation: Model with selected features performed better!")
elif accuracy_selected_features < accuracy_all_features:
    print("\nObservation: Model with selected features performed slightly worse, but might be more efficient.")
else:
    print("\nObservation: Model performance is similar with both sets of features.")


Titanic dataset loaded successfully.

Training data shape: (712, 8), Testing data shape: (179, 8)
Target data shape: (712,), Target test data shape: (179,)

--- Step 3: Using Random Forest for Feature Importance ---

Feature Importances from Random Forest (Top 10):
Sex_male      0.273316
Fare          0.272058
Age           0.252745
Pclass        0.078616
SibSp         0.052192
Parch         0.038490
Embarked_S    0.023095
Embarked_Q    0.009488
dtype: float64

--- Step 4: Applying Recursive Feature Elimination (RFE) ---

--- Submission Requirement: Selected Features ---
Selected features by RFE: ['Pclass', 'SibSp', 'Parch', 'Sex_male', 'Embarked_Q', 'Embarked_S']

--- Step 5: Training Models and Comparing Performance ---

Training model with ALL features...
Accuracy with ALL features: 0.8212

Classification Report (ALL features):
              precision    recall  f1-score   support

           0       0.83      0.87      0.85       105
           1       0.80      0.76      0.78     